# 03 · Tag QA Gate: Model Development

In-notebook experiments that motivated the production configuration used by the pipeline (notebook 05):

* **Baseline:** word TF-IDF (1-2 grams) + one-vs-rest logistic regression, 0.5 cut-off.
* **Candidate:** same text features + draft-time tabular features (price, age rating, platforms, length),
  class-balanced logistic regression, and **per-tag thresholds** tuned on validation.
* **Gate:** a draft is approved when the predicted tag set agrees with the game's real tags (set-F1 >= gate threshold).
  The threshold is chosen to keep **false approvals <= 5%** while maximising auto-approval.

Character n-grams were also tried; they were about 20x slower to train with no lift, so they were dropped.

In [ ]:
# Pin the SageMaker SDK v2 API used by this project (no-op if already installed)
%pip install -q "sagemaker>=2.232,<3"

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
def find_repo_root():
    """The notebooks import from src/, so the whole repo must be cloned, not just notebooks/."""
    candidates = [os.path.expanduser("~/steam-store-copilot")]
    try:  # the kernel's working directory may have been deleted (e.g. after re-cloning)
        candidates = [os.getcwd(), os.path.dirname(os.getcwd())] + candidates
    except FileNotFoundError:
        os.chdir(os.path.expanduser("~"))
    for path in candidates:
        if os.path.isdir(os.path.join(path, "src")) and os.path.exists(os.path.join(path, "src", "config.py")):
            return path
    raise RuntimeError("Could not find the repo's src/ folder. Clone the full repository in Studio "
                       "(Git > Clone Repository) and open the notebooks from steam-store-copilot/notebooks/.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

import boto3
import sagemaker

from src.config import S3Layout

# Works both inside SageMaker Studio and on a laptop with Learner Lab credentials in ~/.aws/credentials
boto_session = boto3.Session(region_name=boto3.Session().region_name or "us-east-1")
sess = sagemaker.Session(boto_session=boto_session)
region = sess.boto_region_name
try:
    role = sagemaker.get_execution_role(sagemaker_session=sess)
except Exception:  # running outside SageMaker: use the Learner Lab execution role
    account = boto_session.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/LabRole"
layout = S3Layout(bucket=sess.default_bucket())
sm = boto3.client("sagemaker", region_name=region)
print("Region:", region)
print("Role:  ", role)
print("Bucket:", layout.bucket)

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "models", "tag_gate"))
import model_lib as ml
from src.data.preprocess import labels_matrix, split_for
from src.data.s3io import read_csv

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})
snap = read_csv(f"{layout.snapshot}/snapshot.csv", keep_default_na=False)
snap["release_date"] = pd.to_datetime(snap["release_date"], errors="coerce")
snap["split"] = [split_for(a, d) for a, d in zip(snap["app_id"], snap["release_date"])]
snap["tag_list"] = snap["tags_22"].map(lambda v: v.split("|") if v else [])
parts = {s: snap[snap["split"] == s] for s in ("train", "val", "test")}
Y = {s: labels_matrix(d["tag_list"]) for s, d in parts.items()}
X = {s: ml.prepare_features(d) for s, d in parts.items()}
{s: len(d) for s, d in parts.items()}

In [ ]:
results = {}
for kind in ("baseline", "improved"):
    start = time.time()
    model = ml.build_pipeline(kind).fit(X["train"], Y["train"])
    p_val, p_test = model.predict_proba(X["val"]), model.predict_proba(X["test"])
    th = np.full(22, 0.5) if kind == "baseline" else ml.tune_thresholds(Y["val"], p_val)
    results[kind] = {"model": model, "p_val": p_val, "p_test": p_test, "th": th,
                     "metrics": ml.tag_metrics(Y["test"], p_test >= th), "fit_s": time.time() - start}
    print(f"{kind:<9} fit {results[kind]['fit_s']:.0f}s  test macro-F1 {results[kind]['metrics']['macro_f1']:.3f}"
          f"  micro-F1 {results[kind]['metrics']['micro_f1']:.3f}")

base_tuned = ml.tag_metrics(Y["test"], results["baseline"]["p_test"] >=
                            ml.tune_thresholds(Y["val"], results["baseline"]["p_val"]))["macro_f1"]
print(f"(ablation) baseline + tuned thresholds: {base_tuned:.3f}")

In [ ]:
summary = pd.Series({
    "Baseline (0.5 cut-off)": results["baseline"]["metrics"]["macro_f1"],
    "Baseline + tuned thresholds": base_tuned,
    "Candidate (tabular + balanced + tuned)": results["improved"]["metrics"]["macro_f1"],
})
ax = summary.plot.barh(color=["#999", "#6a9fd4", "#1f4e8c"], xlim=(0, 0.8))
for i, v in enumerate(summary):
    ax.text(v, i, f" {v:.3f}", va="center")
ax.set_title("Test macro-F1 across 22 tags")
plt.show()

## Per-tag precision and recall

In [ ]:
per_tag = pd.DataFrame({
    kind: {t: m["f1"] for t, m in results[kind]["metrics"]["per_tag"].items()} for kind in results
}).sort_values("improved")
per_tag.plot.barh(figsize=(9, 9), color=["#bbb", "#1f4e8c"], title="Per-tag F1 (test)")
plt.show()
cand = pd.DataFrame(results["improved"]["metrics"]["per_tag"]).T
cand["threshold"] = results["improved"]["th"]
cand.round(3).sort_values("f1", ascending=False)

## Gate operating point
"Mismatched" drafts pair a description with a different game's tags; they simulate copy written for the wrong game.
The chosen threshold maximises auto-approval subject to a 5% false-approval ceiling.

In [ ]:
rng = np.random.default_rng(42)


def mismatch(y):
    perm = rng.permutation(len(y))
    return y[np.where(perm == np.arange(len(y)), (perm + 1) % len(y), perm)]


r = results["improved"]
pred_val = (r["p_val"] >= r["th"]).astype(int)
sg, sm_ = ml.agreement(pred_val, Y["val"]), ml.agreement(pred_val, mismatch(Y["val"]))
grid = np.round(np.arange(0, 1.001, 0.01), 2)
curve = pd.DataFrame([ml.gate_rates(sg, sm_, t) for t in grid], index=grid)
gate_t, rates = ml.tune_gate_threshold(sg, sm_, 0.05)

fig, ax = plt.subplots(figsize=(10, 4))
curve.plot(ax=ax)
ax.axvline(gate_t, color="k", ls="--", label=f"chosen threshold {gate_t}")
ax.axhline(0.05, color="red", ls=":", label="5% false-approval ceiling")
ax.set_xlabel("gate threshold (set-F1 agreement)")
ax.legend()
ax.set_title("Auto-approval vs false-approval (validation)")
plt.show()

pred_test = (r["p_test"] >= r["th"]).astype(int)
test_rates = ml.gate_rates(ml.agreement(pred_test, Y["test"]), ml.agreement(pred_test, mismatch(Y["test"])), gate_t)
print("gate threshold:", gate_t, "| validation:", rates, "| test:", test_rates)

In [ ]:
examples = parts["test"].head(8).copy()
examples["predicted"] = [", ".join(t for t, p in zip(ml.TAGS, row) if p) for row in pred_test[:8]]
examples["true"] = examples["tags_22"].str.replace("|", ", ")
examples["agreement"] = ml.agreement(pred_test[:8], Y["test"][:8]).round(2)
examples["decision"] = ml.gate_decisions(examples["agreement"].values, gate_t)
pd.set_option("display.max_colwidth", 80)
examples[["app_id", "true", "predicted", "agreement", "decision"]]

## Conclusion
The candidate beats the baseline by roughly **+0.13 macro-F1**. Most of that lift comes from per-tag thresholds,
which is an honest finding: in this problem, calibration matters more than model capacity. The model stays small
(about 30 s to train), so retraining on drift is cheap. The same code (`src/models/tag_gate`) runs in the SageMaker
Pipeline.